In [1]:
# ============================================
# Stage 3 – Dual-Token Fusion (RGB + FFT)
# Full fine-tuning with DeepfakeBench metrics
# ============================================

import os, random, torch, torch.nn as nn, torch.optim as optim
import numpy as np
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import torch.fft as fft
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, roc_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d
from PIL import Image
from io import BytesIO
import warnings
warnings.filterwarnings("ignore")


# ---- DeepfakeBench metric helper ----
def calc_metrics(y_true, y_prob):
    y_pred = (y_prob > 0.5).astype(int)
    auc  = roc_auc_score(y_true, y_prob)
    f1   = f1_score(y_true, y_pred)
    acc  = accuracy_score(y_true, y_pred)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    eer  = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)
    return auc, f1, eer, acc



In [2]:
# ============================================
#  Frequency Magnitude  +  Dual-Token Transformer
# ============================================

class FFTMagnitude(nn.Module):
    """Compute magnitude spectrum (log-scaled) of feature maps."""
    def forward(self, x):
        # always compute FFT in float32 for stability
        x = x.float()
        f = torch.fft.fft2(x)
        f = torch.abs(torch.fft.fftshift(f))
        return torch.log1p(f)


class Stage3Hybrid(nn.Module):
    def __init__(self, embed_dim=512, num_heads=8, depth=6, num_classes=2):
        super().__init__()
        # ---- RGB backbone ----
        self.resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.resnet = nn.Sequential(*list(self.resnet.children())[:-2])   # remove avgpool+fc
        self.fft_layer = FFTMagnitude()

        # ---- Projections ----
        self.proj_rgb = nn.Conv2d(2048, embed_dim, 1)
        self.proj_fft = nn.Conv2d(2048, embed_dim, 1)

        # ---- Positional embeddings ----
        self.pos_embed = nn.Parameter(torch.zeros(1, 200, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        # ---- Transformer encoder ----
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads,
                                                   dim_feedforward=embed_dim*4,
                                                   dropout=0.1, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)

        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        rgb_feat = self.resnet(x)                  # [B,2048,H,W]
        fft_feat = self.fft_layer(rgb_feat)        # [B,2048,H,W]

        rgb = self.proj_rgb(rgb_feat)              # [B,embed,H,W]
        fftm = self.proj_fft(fft_feat)
        fused = torch.cat([rgb, fftm], dim=2)      # concat along H → 200 tokens (100 RGB + 100 FFT)

        tokens = fused.flatten(2).transpose(1, 2)  # [B,N,embed]
        tokens = tokens + self.pos_embed[:, :tokens.size(1)]
        tokens = self.transformer(tokens)
        out = tokens.mean(dim=1)
        return self.head(out)


In [3]:
def train_stage3():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = Stage3Hybrid().to(device)

    # ---- Unfreeze everything ----
    for p in model.parameters():
        p.requires_grad = True
    model.train()

    # ---- Random JPEG degradation ----
    def random_jpeg(img):
        buf = BytesIO()
        img.save(buf, format="JPEG", quality=random.randint(40, 90))
        buf.seek(0)
        return Image.open(buf)

    # ---- Transforms ----
    train_tfms = transforms.Compose([
        transforms.Resize((320,320)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.2,0.2,0.1,0.05),
        transforms.RandomApply([transforms.Lambda(random_jpeg)], p=0.3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])
    val_tfms = transforms.Compose([
        transforms.Resize((320,320)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])

    # ---- Datasets ----
    train_path = "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train"
    val_path   = "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/valid"
    test_path  = "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/test"

    trainset = datasets.ImageFolder(train_path, transform=train_tfms)
    valset   = datasets.ImageFolder(val_path, transform=val_tfms)
    testset  = datasets.ImageFolder(test_path, transform=val_tfms)

    real_idx = trainset.class_to_idx['real']  # ensures correct probability column

    trainloader = DataLoader(trainset, batch_size=16, shuffle=True, num_workers=2)
    valloader   = DataLoader(valset, batch_size=16, shuffle=False, num_workers=2)
    testloader  = DataLoader(testset, batch_size=16, shuffle=False, num_workers=2)

    # ---- Optimizer / loss / scaler ----
    opt = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-5)
    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler("cuda")

    epochs = 10
    best_auc, best_acc = 0.0, 0.0

    for epoch in range(epochs):
        model.train()
        running_loss = 0
        for imgs, lbls in tqdm(trainloader, desc=f"Epoch {epoch+1}/{epochs}", ncols=100):
            imgs, lbls = imgs.to(device), lbls.to(device)
            opt.zero_grad()
            with torch.amp.autocast("cuda"):
                out = model(imgs)
                loss = criterion(out, lbls)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            running_loss += loss.item()

        # ---- Validation ----
        model.eval(); y_true, y_prob = [], []
        with torch.no_grad():
            for imgs, lbls in valloader:
                imgs = imgs.to(device)
                probs = torch.softmax(model(imgs), dim=1)[:, real_idx].cpu().numpy()
                y_true.extend(lbls.numpy())
                y_prob.extend(probs)

        auc, f1, eer, acc = calc_metrics(np.array(y_true), np.array(y_prob))
        print(f"Epoch {epoch+1}: loss={running_loss/len(trainloader):.4f} | "
              f"AUROC={auc:.3f} | F1={f1:.3f} | EER={eer:.3f} | ACC={acc:.3f}")

        torch.save(model.state_dict(), f"/kaggle/working/stage3_epoch{epoch+1}.pth")
        if auc > best_auc:
            best_auc, best_acc = auc, acc
            torch.save(model.state_dict(), "/kaggle/working/best_auc_stage3.pth")
            print(f"★ New best AUC {best_auc:.3f} (ACC={best_acc:.3f})")

    print("\n Training complete!")
    print(f" Best model achieved: AUROC = {best_auc:.3f}, Accuracy = {best_acc:.3f}")
    print(" Saved as: /kaggle/working/best_auc_stage3.pth")

    return model, testloader, real_idx


In [4]:
def evaluate(model, testloader, real_idx, device="cuda"):
    model.eval(); y_true, y_prob = [], []
    with torch.no_grad():
        for imgs, lbls in tqdm(testloader, desc="Testing", ncols=100):
            imgs = imgs.to(device)
            probs = torch.softmax(model(imgs), dim=1)[:, real_idx].cpu().numpy()
            y_true.extend(lbls.numpy())
            y_prob.extend(probs)
    auc, f1, eer, acc = calc_metrics(np.array(y_true), np.array(y_prob))
    print(f"\n Test Results → AUROC={auc:.3f} | F1={f1:.3f} | EER={eer:.3f} | ACC={acc:.3f}")


In [5]:
if __name__ == "__main__":
    model, testloader, real_idx = train_stage3()
    evaluate(model, testloader, real_idx)


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 192MB/s]
Epoch 1/10: 100%|███████████████████████████████████████████████| 6250/6250 [26:40<00:00,  3.90it/s]


Epoch 1: loss=0.6977 | AUROC=0.508 | F1=0.000 | EER=0.494 | ACC=0.500
★ New best AUC 0.508 (ACC=0.500)


Epoch 2/10: 100%|███████████████████████████████████████████████| 6250/6250 [26:48<00:00,  3.89it/s]


Epoch 2: loss=0.6935 | AUROC=0.500 | F1=0.000 | EER=0.500 | ACC=0.500


Epoch 3/10: 100%|███████████████████████████████████████████████| 6250/6250 [26:47<00:00,  3.89it/s]


...

Epoch 9: loss=0.6933 | AUROC=0.500 | F1=0.667 | EER=0.500 | ACC=0.500


Epoch 10/10: 100%|██████████████████████████████████████████████| 6250/6250 [26:48<00:00,  3.88it/s]


Epoch 10: loss=0.6932 | AUROC=0.500 | F1=0.000 | EER=0.500 | ACC=0.500

 Training complete!
 Best model achieved: AUROC = 0.508, Accuracy = 0.500
 Saved as: /kaggle/working/best_auc_stage3.pth


Testing: 100%|██████████████████████████████████████████████████| 1250/1250 [01:51<00:00, 11.22it/s]


 Test Results → AUROC=0.500 | F1=0.000 | EER=0.500 | ACC=0.500
